# Load MEXT Education Content Data

This notebook loads the **Children's Learning Support Site Content Information** (子供の学び応援サイト掲載コンテンツ情報) CSV into a Delta table.

- Source: [e-Gov Data Portal](https://data.e-gov.go.jp/data/en/dataset/mext_20210222_0025)
- Publisher: Ministry of Education, Culture, Sports, Science and Technology (MEXT / 文部科学省)
- License: CC BY 4.0

The source CSV is Shift-JIS encoded with trailing empty columns. We use Python's csv module
to parse it correctly, then create a Spark DataFrame and write to a Delta table.

In [ ]:
import csv
from pyspark.sql.types import StructType, StructField, StringType

# Read CSV with Python's csv module (handles Shift-JIS encoding and
# embedded newlines correctly, unlike spark.read.csv)
csv_path = '/lakehouse/default/Files/mext/mext_education_content.csv'
rows = []
with open(csv_path, 'r', encoding='shift_jis', errors='replace') as f:
    reader = csv.reader(f)
    header = next(reader)
    # Keep only meaningful columns (source has trailing empty columns)
    header = [h.strip() for h in header[:15]]
    for row in reader:
        vals = [v.strip() for v in row[:15]]
        if vals[0]:  # skip empty rows
            rows.append(vals)

print(f'Read {len(rows)} data rows')
print(f'Columns: {header}')

In [ ]:
# Create Spark DataFrame and write to Delta table
spark.sql('CREATE SCHEMA IF NOT EXISTS mext')

schema = StructType([StructField(c, StringType(), True) for c in header])
df = spark.createDataFrame(rows, schema=schema)

df.write.format('delta').mode('overwrite').saveAsTable('mext.education_content')

count = spark.table('mext.education_content').count()
print(f'Table mext.education_content has {count} rows')

In [ ]:
# Show subject breakdown
print('Rows by subject (教科):')
spark.sql("""
    SELECT `教材_教科等` AS subject, COUNT(*) AS count
    FROM mext.education_content
    GROUP BY `教材_教科等`
    ORDER BY count DESC
""").show(truncate=False)